# Fine-tune and Evaluate OPT-1.3B on CNN/DailyMail

This notebook uses the optimized setup:

### Training and validation

```text
Complete article that fits the preset limit
        ↓
Complete structured target
        ↓
Fine-tune OPT-1.3B
```

### Testing and deployment

```text
Article fits?
  Yes → direct structured generation
  No  → split article → summarize chunks → combine → final structured generation
```

The model is trained to generate four labelled sections:

1. Summary
2. Key Points
3. Specialized Vocabulary Explanation
4. Quiz

## 0. Setup

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import re
import json
import time
from functools import partial

import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from rouge_score import rouge_scorer
import textstat

PROCESSED_DIR = "data/processed"
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-1.3b"
FINETUNED_MODEL_DIR = "models/opt-1.3b-cnn-news-simplifier"
RESULTS_DIR = "results"

MAX_TRAIN_LENGTH = 1536
MAX_TARGET_TOKENS = 320

MAX_CONTEXT = 2048
MAX_NEW_TOKENS = 350
CHUNK_ARTICLE_TOKENS = 700
CHUNK_NEW_TOKENS = 100

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle before running this notebook.")

DEVICE = "cuda"
print(f"Using device: {DEVICE}")

## 1. Load train, validation and test splits

In [ ]:
def load_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. Run preprocessing_cnn.ipynb and "
            "split_data.ipynb first."
        )

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return pd.DataFrame(records)


train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

print(f"Train:      {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test:       {len(test_df)}")
print(
    "Long/non-fitting test records:",
    int((test_df["fits_training_limit"] == False).sum()),
)

## 2. Prepare complete training pairs

Training and validation records were filtered by the split notebook. This cell
checks them again. The dataset class does not silently truncate a record; it
raises an error if a supposedly complete pair exceeds the preset limit.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def verify_complete_pairs(frame: pd.DataFrame, split_name: str):
    invalid = frame[
        (frame["fits_training_limit"] != True) |
        (frame["target_token_length"] > MAX_TARGET_TOKENS) |
        (frame["total_training_tokens"] > MAX_TRAIN_LENGTH)
    ]

    print(
        f"{split_name}: {len(frame) - len(invalid)}/{len(frame)} "
        "complete pairs fit the limits"
    )

    if not invalid.empty:
        raise ValueError(
            f"{split_name} contains {len(invalid)} non-fitting records. "
            "Run the updated split_data.ipynb again."
        )


verify_complete_pairs(train_df, "Training")
verify_complete_pairs(val_df, "Validation")


class NewsSimplificationDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
        max_length: int = MAX_TRAIN_LENGTH,
        max_target_tokens: int = MAX_TARGET_TOKENS,
    ):
        self.prompts = frame["prompt"].tolist()
        self.targets = frame["training_target"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_target_tokens = max_target_tokens

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        prompt = str(self.prompts[index])
        target = str(self.targets[index]).strip() + self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]

        target_ids = self.tokenizer(
            target,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        if len(target_ids) > self.max_target_tokens:
            raise ValueError(
                f"Target {index} has {len(target_ids)} tokens; "
                f"maximum is {self.max_target_tokens}."
            )

        if len(prompt_ids) + len(target_ids) > self.max_length:
            raise ValueError(
                f"Training pair {index} has "
                f"{len(prompt_ids) + len(target_ids)} tokens; "
                f"maximum is {self.max_length}."
            )

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


def collate_fn(batch, pad_token_id: int):
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        pad_length = max_len - len(item["input_ids"])

        input_ids.append(
            item["input_ids"] + [pad_token_id] * pad_length
        )
        attention_masks.append(
            item["attention_mask"] + [0] * pad_length
        )
        labels.append(
            item["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


train_dataset = NewsSimplificationDataset(train_df, tokenizer)
val_dataset = NewsSimplificationDataset(val_df, tokenizer)

print(
    f"Prepared {len(train_dataset)} training examples and "
    f"{len(val_dataset)} validation examples"
)

## 3. Load OPT-1.3B and fine-tune with QLoRA

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
)

print("Model context limit:", model.config.max_position_embeddings)

if MAX_TRAIN_LENGTH > model.config.max_position_embeddings:
    raise ValueError(
        "MAX_TRAIN_LENGTH exceeds the model context limit."
    )

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(FINETUNED_MODEL_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to=[],
)

safe_collator = partial(
    collate_fn,
    pad_token_id=tokenizer.pad_token_id,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=safe_collator,
)

train_result = trainer.train()
print(train_result)

## 4. Save the fine-tuned adapter

In [ ]:
trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)
print(f"Saved model adapter to '{FINETUNED_MODEL_DIR}'")

## 5. Direct or chunk-based test generation

- A short article is generated directly.
- A long article is split by Python into sentence-based chunks.
- OPT summarizes each chunk.
- Python combines those summaries.
- OPT produces the final four-section output.

The model does not learn the splitting operation; Python performs it.

In [ ]:
with open(TEMPLATES_PATH, "r", encoding="utf-8") as f:
    prompt_templates = json.load(f)

single_prompt_template = prompt_templates["single_prompt_template"]
chunk_prompt_template = prompt_templates["chunk_prompt_template"]
final_prompt_template = prompt_templates["final_prompt_template"]

finetuned_model = trainer.model
finetuned_model.eval()
finetuned_model.config.use_cache = True

SECTION_HEADERS = [
    "Key Points:",
    "Specialized Vocabulary Explanation:",
    "Quiz:",
]

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def token_length(text: str) -> int:
    return len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )


def extract_summary_section(generated_text: str) -> str:
    text = generated_text.strip()

    if text.startswith("Summary:"):
        text = text[len("Summary:"):].strip()

    cut_positions = [
        text.find(header)
        for header in SECTION_HEADERS
        if text.find(header) != -1
    ]

    if cut_positions:
        text = text[:min(cut_positions)]

    return text.strip()


def split_article_into_chunks(
    article: str,
    max_article_tokens: int = CHUNK_ARTICLE_TOKENS,
):
    sentences = [
        sentence.strip()
        for sentence in SENTENCE_SPLIT_RE.split(article.strip())
        if sentence.strip()
    ]

    chunks = []
    current_sentences = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = token_length(sentence)

        if (
            current_sentences and
            current_tokens + sentence_tokens > max_article_tokens
        ):
            chunks.append(" ".join(current_sentences))
            current_sentences = []
            current_tokens = 0

        # Safety fallback for one unusually long sentence.
        if sentence_tokens > max_article_tokens:
            sentence_ids = tokenizer(
                sentence,
                add_special_tokens=False,
                truncation=False,
            )["input_ids"]

            for start in range(0, len(sentence_ids), max_article_tokens):
                piece_ids = sentence_ids[start:start + max_article_tokens]
                piece = tokenizer.decode(
                    piece_ids,
                    skip_special_tokens=True,
                ).strip()

                if current_sentences:
                    chunks.append(" ".join(current_sentences))
                    current_sentences = []
                    current_tokens = 0

                if piece:
                    chunks.append(piece)
            continue

        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks if chunks else [article]


def generate_response(
    prompt: str,
    max_new_tokens: int,
):
    max_input_tokens = MAX_CONTEXT - max_new_tokens

    inputs = tokenizer(
        prompt,
        truncation=True,
        max_length=max_input_tokens,
        return_tensors="pt",
    ).to(DEVICE)

    input_length = inputs["input_ids"].shape[1]

    start_time = time.time()

    with torch.no_grad():
        output_ids = finetuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.time() - start_time

    generated_ids = output_ids[0][input_length:]
    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return generated_text.strip(), elapsed


def build_final_prompt(chunk_summaries):
    combined = "\n\n".join(
        f"Part {index}: {summary}"
        for index, summary in enumerate(chunk_summaries, start=1)
    )

    prompt = final_prompt_template.format(
        chunk_summaries=combined
    )

    max_input_tokens = MAX_CONTEXT - MAX_NEW_TOKENS

    if token_length(prompt) <= max_input_tokens:
        return prompt

    empty_prompt = final_prompt_template.format(chunk_summaries="")
    available_tokens = max(
        1,
        max_input_tokens - token_length(empty_prompt) - 10,
    )

    combined_ids = tokenizer(
        combined,
        add_special_tokens=False,
        truncation=True,
        max_length=available_tokens,
    )["input_ids"]

    shortened_combined = tokenizer.decode(
        combined_ids,
        skip_special_tokens=True,
    )

    return final_prompt_template.format(
        chunk_summaries=shortened_combined
    )


def generate_article_output(article: str):
    direct_prompt = single_prompt_template.format(article=article)
    direct_limit = MAX_CONTEXT - MAX_NEW_TOKENS

    if token_length(direct_prompt) <= direct_limit:
        output, elapsed = generate_response(
            direct_prompt,
            max_new_tokens=MAX_NEW_TOKENS,
        )
        return output, elapsed, "direct", 1

    chunks = split_article_into_chunks(article)
    chunk_summaries = []
    total_time = 0.0

    for chunk in chunks:
        chunk_prompt = chunk_prompt_template.format(chunk=chunk)
        chunk_output, chunk_time = generate_response(
            chunk_prompt,
            max_new_tokens=CHUNK_NEW_TOKENS,
        )

        chunk_summary = extract_summary_section(chunk_output)
        if chunk_summary:
            chunk_summaries.append(chunk_summary)

        total_time += chunk_time

    final_prompt = build_final_prompt(chunk_summaries)
    final_output, final_time = generate_response(
        final_prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    total_time += final_time

    return final_output, total_time, "chunked", len(chunks)

In [ ]:
generated_texts = []
generation_times = []
generation_methods = []
chunk_counts = []

for index, row in test_df.iterrows():
    output, elapsed, method, chunk_count = generate_article_output(
        row["article"]
    )

    generated_texts.append(output)
    generation_times.append(elapsed)
    generation_methods.append(method)
    chunk_counts.append(chunk_count)

    if (index + 1) % 10 == 0 or (index + 1) == len(test_df):
        print(f"Generated {index + 1}/{len(test_df)} responses")


test_df = test_df.copy()
test_df["generated_text"] = generated_texts
test_df["generation_time_sec"] = generation_times
test_df["generation_method"] = generation_methods
test_df["generation_chunk_count"] = chunk_counts
test_df["generated_summary"] = test_df["generated_text"].apply(
    extract_summary_section
)

test_df[
    [
        "generation_method",
        "generation_chunk_count",
        "summary",
        "generated_summary",
        "generated_text",
    ]
].head()

## 6. Evaluate

ROUGE compares only the generated **Summary** section with the original
CNN/DailyMail highlights. It measures reference overlap, not complete factual
accuracy.

The structured completion flag checks whether all three additional headings
appear in the output.

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)


def compute_rouge(reference: str, hypothesis: str):
    scores = scorer.score(reference, hypothesis)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def word_count(text: str) -> int:
    return max(1, len(str(text).split()))


def has_complete_structure(text: str) -> bool:
    return all(header in text for header in SECTION_HEADERS)


rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
flesch_scores = []
compression_ratios = []
structured_complete = []

for _, row in test_df.iterrows():
    rouge = compute_rouge(
        row["summary"],
        row["generated_summary"],
    )

    rouge1_scores.append(rouge["rouge1"])
    rouge2_scores.append(rouge["rouge2"])
    rougeL_scores.append(rouge["rougeL"])

    flesch_scores.append(
        textstat.flesch_reading_ease(row["generated_text"])
        if row["generated_text"] else 0.0
    )

    compression_ratios.append(
        word_count(row["generated_summary"]) /
        word_count(row["article"])
    )

    structured_complete.append(
        has_complete_structure(row["generated_text"])
    )


test_df["rouge1"] = rouge1_scores
test_df["rouge2"] = rouge2_scores
test_df["rougeL"] = rougeL_scores
test_df["flesch_reading_ease"] = flesch_scores
test_df["compression_ratio"] = compression_ratios
test_df["structured_output_complete"] = structured_complete

## 7. Aggregate and save results

In [ ]:
summary_metrics = pd.DataFrame({
    "metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Flesch Reading Ease",
        "Compression Ratio",
        "Generation Time (sec)",
        "Structured Output Completion Rate",
    ],
    "mean": [
        test_df["rouge1"].mean(),
        test_df["rouge2"].mean(),
        test_df["rougeL"].mean(),
        test_df["flesch_reading_ease"].mean(),
        test_df["compression_ratio"].mean(),
        test_df["generation_time_sec"].mean(),
        test_df["structured_output_complete"].mean(),
    ],
    "std": [
        test_df["rouge1"].std(),
        test_df["rouge2"].std(),
        test_df["rougeL"].std(),
        test_df["flesch_reading_ease"].std(),
        test_df["compression_ratio"].std(),
        test_df["generation_time_sec"].std(),
        test_df["structured_output_complete"].std(),
    ],
})

print(summary_metrics.to_string(index=False))

print("\nGeneration methods:")
print(test_df["generation_method"].value_counts())

per_example_path = os.path.join(
    RESULTS_DIR,
    "opt13b_cnn_test_results.csv",
)
summary_path = os.path.join(
    RESULTS_DIR,
    "opt13b_cnn_evaluation_summary.csv",
)

columns_to_drop = [
    "prompt",
    "training_target",
    "key_points",
    "vocabulary",
    "quiz",
]

test_df.drop(
    columns=columns_to_drop,
    errors="ignore",
).to_csv(per_example_path, index=False)

summary_metrics.to_csv(summary_path, index=False)

print(f"\nSaved test results to '{per_example_path}'")
print(f"Saved evaluation summary to '{summary_path}'")